<a href="https://colab.research.google.com/github/adityamahajan5411/adityamahajan5411-AnalyticsEngineering/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

To get a better understanding of the data provided, I requested a python code that summarises each table and gives a list of the columns. This is so that I can get an overview of the data and choose the columns of importance.
To find inconsistencies in the data, I requested for a code that can find missing (null values) data, duplicates, etc. to make the process of cleaning easier. Prompt: Write a python code to summarise each table in the given data, provide the columns, identify anything that looks odd, missing, or inconsistent

=== Summary for table: 'trips' ===
Shape: 612 rows, 7 columns

--- Columns & Data Types ---
                    dtype  non_null_count  null_count  null_pct  unique_values
trip_id            object             612           0       0.0            600
bike_id            object             612           0       0.0             60
start_station_id   object             612           0       0.0             12
end_station_id     object             612           0       0.0             12
start_time         object             612           0       0.0            597
duration_min      float64             582          30       4.9             45
user_type          object             612           0       0.0              7


Duplicate primary key values ('trip_id'): 12
Entirely duplicate rows: 12

--- Descriptive Statistics ---


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
count,612,612,612,612,612,582.000000,612
unique,600,60,12,12,597,NaN,7
top,T0449,B051,S02,S01,2025-03-06 08:14:00,NaN,Member
freq,2,21,87,94,2,NaN,331
mean,NaN,NaN,NaN,NaN,NaN,27.891753,NaN
std,NaN,NaN,NaN,NaN,NaN,145.064305,NaN
min,NaN,NaN,NaN,NaN,NaN,4.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,12.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,16.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,22.000000,NaN



=== Summary for table: 'bikes' ===
Shape: 60 rows, 3 columns

--- Columns & Data Types ---
                dtype  non_null_count  null_count  null_pct  unique_values
bike_id        object              60           0       0.0             60
bike_type      object              60           0       0.0              7
purchase_year   int64              60           0       0.0              4


Duplicate primary key values ('bike_id'): 0
Entirely duplicate rows: 0

--- Descriptive Statistics ---


,bike_id,bike_type,purchase_year
count,60,60,60.000000
unique,60,7,NaN
top,B001,Standard,NaN
freq,1,36,NaN
mean,NaN,NaN,2022.550000
std,NaN,NaN,1.185005
min,NaN,NaN,2021.000000
25%,NaN,NaN,2021.000000
50%,NaN,NaN,2023.000000
75%,NaN,NaN,2024.000000



=== Summary for table: 'stations' ===
Shape: 12 rows, 4 columns

--- Columns & Data Types ---
                dtype  non_null_count  null_count  null_pct  unique_values
station_id     object              12           0       0.0             12
station_name   object              12           0       0.0             12
zone           object              12           0       0.0              5
capacity      float64              11           1       8.3             10


Duplicate primary key values ('station_id'): 0
Entirely duplicate rows: 0

--- Descriptive Statistics ---


,station_id,station_name,zone,capacity
count,12,12,12,11.000000
unique,12,12,5,NaN
top,S01,Market Square,Centre,NaN
freq,1,1,5,NaN
mean,NaN,NaN,NaN,22.909091
std,NaN,NaN,NaN,5.539938
min,NaN,NaN,NaN,14.000000
25%,NaN,NaN,NaN,19.000000
50%,NaN,NaN,NaN,24.000000
75%,NaN,NaN,NaN,27.500000


Once the results suggested that there are duplicates in the 'trips' table, I wanted to examine the exact rows to identify whether they are all duplicates of 1 row or multiple. This will guide the decision to delete duplicate rows.
Prompt: show me the duplicate rows in table 'trips'

In [4]:
# Show duplicate rows in the trips table
trips_df = tables['trips']

# Find rows where the entire row is duplicated (keeping all occurrences of duplicates to compare)
duplicates = trips_df[trips_df.duplicated(keep=False)]
print(f"Found {len(duplicates)} rows that are duplicated (including their originals):")
display(duplicates.sort_values(by='trip_id'))

Found 24 rows that are duplicated (including their originals):


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
96,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
97,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
175,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
176,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
185,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
186,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
285,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
286,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
327,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member


## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

In [6]:
# 2. Clean and Prepare Data
import pandas as pd
import numpy as np

# Create working copies of the dataframes
clean_trips = tables['trips'].copy()
clean_bikes = tables['bikes'].copy()
clean_stations = tables['stations'].copy()

print(f"Original trips shape: {clean_trips.shape}")

# 1. Remove duplicate rows in trips
clean_trips = clean_trips.drop_duplicates()
print(f"Trips shape after removing duplicates: {clean_trips.shape}")

# 2. Standardise 'user_type' in trips to Title Case ('Member' / 'Casual')
clean_trips['user_type'] = clean_trips['user_type'].str.strip().str.title()
print(f"Standardised user types: {clean_trips['user_type'].unique()}")

# 3. Standardise 'bike_type' in bikes to Title Case ('Standard' / 'Electric') and merge synonyms
clean_bikes['bike_type'] = clean_bikes['bike_type'].str.strip().str.title()
clean_bikes['bike_type'] = clean_bikes['bike_type'].replace({'E-Bike': 'Electric'})
print(f"Standardised bike types: {clean_bikes['bike_type'].unique()}")

# 4. Standardise start_time dates to datetime (handling mixed YYYY-MM-DD and DD/MM/YYYY formats)
clean_trips['start_time'] = pd.to_datetime(clean_trips['start_time'], errors='coerce', format='mixed')
print(f"Missing dates after parsing: {clean_trips['start_time'].isnull().sum()}")

# 5. Handle missing values:
# - We preserve missing 'duration_min' as NaN so they don't skew our average calculations.
# - We preserve missing 'capacity' in stations as NaN since capacity isn't needed for calculating trip duration.
print(f"Missing durations in clean_trips: {clean_trips['duration_min'].isnull().sum()}")
print(f"Missing capacity in clean_stations: {clean_stations['capacity'].isnull().sum()}")

# Save the cleaned datasets back to tables dictionary and register to DuckDB
tables['clean_trips'] = clean_trips
tables['clean_bikes'] = clean_bikes
tables['clean_stations'] = clean_stations

con.register('clean_trips', clean_trips)
con.register('clean_bikes', clean_bikes)
con.register('clean_stations', clean_stations)
print("\nCleaned tables successfully registered to DuckDB!")

Original trips shape: (612, 7)
Trips shape after removing duplicates: (600, 7)
Standardised user types: ['Casual' 'Member']
Standardised bike types: ['Standard' 'Electric']
Missing dates after parsing: 0
Missing durations in clean_trips: 30
Missing capacity in clean_stations: 1

Cleaned tables successfully registered to DuckDB!


## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

## 6. What I could not verify

*What remains unchecked? What would you do next?*